# ScamShield OSS: Kaggle backend

**Before you start:** Accelerator = **GPU T4 x2**, Internet = **On**, and add a Kaggle Secret named `NGROK_TOKEN`
(Add-ons -> Secrets) with your ngrok authtoken, attached to this notebook.

Run the cells **in order, one at a time**, and wait for each to finish. Do not interrupt a running cell.

## 1. System packages and Python libraries

In [ ]:
!apt-get update -y
!apt-get install -y zstd pciutils lshw ffmpeg
!which zstd
!pip install -q faster-whisper fastapi uvicorn pyngrok python-multipart ollama pydantic pillow sqlmodel requests

## 2. Install Ollama (needs zstd from cell 1)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama

## 3. Start Ollama (detached, with auto-restart) and pull the models

Uses `127.0.0.1`, not `0.0.0.0`; the Python client cannot connect to `0.0.0.0`.
The first run downloads about 11 GB, so it takes several minutes.

In [ ]:
import subprocess, time, os, threading, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"   # keep models in VRAM -> no reload lag during the demo

def ollama_up():
    try:
        return requests.get("http://127.0.0.1:11434", timeout=2).ok
    except Exception:
        return False

def start_ollama():
    log = open("/kaggle/working/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log,
                     env=os.environ.copy(), start_new_session=True)  # survives cell interrupts

def watchdog():
    while True:
        if not ollama_up():
            start_ollama()
            time.sleep(10)
        time.sleep(5)

if not ollama_up():
    start_ollama()
    time.sleep(10)
threading.Thread(target=watchdog, daemon=True).start()
print("Ollama up:", ollama_up())

!ollama pull qwen2.5vl:7b
!ollama pull qwen3:8b
!ollama list

## 4. Get the code from GitHub

Edit `REPO_URL` to point at your repo (it must be public, or use a token you keep in a Kaggle Secret).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/scamshield-oss.git"   # <- change this

%cd /kaggle/working
!rm -rf scamshield-oss
!git clone {REPO_URL}
!ls scamshield-oss/backend

## 5. Start the API and open the public tunnel

Prints your public URL; paste it into the dashboard. The server runs in a background thread,
so this cell finishes while the API keeps running. Re-running it is safe.

In [ ]:
import os, sys, time, threading, requests, uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

os.environ["OLLAMA_URL"] = "http://127.0.0.1:11434"
os.chdir("/kaggle/working/scamshield-oss/backend")
sys.path.insert(0, os.getcwd())

from main import app          # loads Whisper onto the GPU (takes a minute)

# --- ngrok: reset any stale config/tunnels, then set the token from the Kaggle Secret
ngrok.kill()
cfg = "/root/.config/ngrok/ngrok.yml"
if os.path.exists(cfg):
    os.remove(cfg)
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))

# --- API in a background thread (uvicorn.run() cannot be called inside a notebook loop)
try:
    server
    server.should_exit = True
    time.sleep(2)
except NameError:
    pass
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
threading.Thread(target=server.run, daemon=True).start()

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            break
    except Exception:
        time.sleep(1)
print("Local check:", requests.get("http://127.0.0.1:8000/health").json())

public_url = ngrok.connect("http://127.0.0.1:8000").public_url
print(f"\nSCAMSHIELD OSS LIVE BACKEND URL: {public_url}\n")

## 6. Smoke test (optional)

Should print JSON with a high `coercion_score`. The first call is slow while the model loads.

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/analyze-call -F "fallback_text=This is a CBI officer. Transfer 50000 rupees now or you will be arrested."